# DPGAN effective-epsilon audit: threshold, orientation, TP/FP per epsilon

Re-computation from saved raw attack scores. 

For each epsilon, for each of the 5 attacks, use TAPAS's `EffectiveEpsilonReport` to 
- select threshold and orientation on validation slice
- count TP/FP on test slice
- compute eps_low, eps_high at 90, 95, 99 Clopper-Pearson CI
- check against saved per_attack eff_eps CSVs. 

num_train/num_test = 1000/2500

### Cell 1: setup and file locations

In [1]:
from pathlib import Path
from types import SimpleNamespace
import warnings

import numpy as np
import pandas as pd
from tapas.report import EffectiveEpsilonReport

warnings.filterwarnings("ignore")
pd.set_option("display.width", 250, "display.max_columns", 60, "display.float_format", lambda x: f"{x:.4f}")

# benchmark_tapas/ = nearest ancestor holding config.py (same trick as the audit scripts)
HERE = Path.cwd().resolve()
BENCH = next(p for p in [HERE, *HERE.parents] if (p / "config.py").exists())
RES = BENCH / "results"

# epsilon -> (raw scores CSV, per-attack effeps CSV), relative to RES
SOURCES = {
    0.1:   ("eps_sweep/dpgan/raw_scores_dpgan_eps0.1.csv",
            "eps_sweep/dpgan/eps0.1/effeps_dpgan_eps0.1.csv"),
    0.999: ("extras/dpgan_spike_diagnosis/privacy/raw_scores_dpgan_eps0.999.csv",
            "extras/dpgan_spike_diagnosis/privacy/eps0.999/effeps_dpgan_eps0.999.csv"),
    1.0:   ("sample_size_sweep/raw_scores_dpgan_1000_2500.csv",
            "sample_size_sweep/dpgan/1000_2500/effeps_dpgan_1000_2500.csv"),
    1.001: ("extras/dpgan_spike_diagnosis/privacy/raw_scores_dpgan_eps1.001.csv",
            "extras/dpgan_spike_diagnosis/privacy/eps1.001/effeps_dpgan_eps1.001.csv"),
    10.0:  ("eps_sweep/dpgan/raw_scores_dpgan_eps10.csv",
            "eps_sweep/dpgan/eps10/effeps_dpgan_eps10.csv"),
    100.0: ("eps_sweep/dpgan/raw_scores_dpgan_eps100.csv",
            "eps_sweep/dpgan/eps100/effeps_dpgan_eps100.csv"),
}
CONF = (0.9, 0.95, 0.99)
print("benchmark_tapas:", BENCH)
for eps, (raw, eff) in SOURCES.items():
    assert (RES / raw).exists() and (RES / eff).exists(), (eps, raw, eff)
    print(f"eps={eps:<6} raw={raw}\n{'':11}effeps={eff}")

benchmark_tapas: /Users/PengOlivia/Desktop/Uni/USYD VRI/priv-sdg/benchmark_tapas
eps=0.1    raw=eps_sweep/dpgan/raw_scores_dpgan_eps0.1.csv
           effeps=eps_sweep/dpgan/eps0.1/effeps_dpgan_eps0.1.csv
eps=0.999  raw=extras/dpgan_spike_diagnosis/privacy/raw_scores_dpgan_eps0.999.csv
           effeps=extras/dpgan_spike_diagnosis/privacy/eps0.999/effeps_dpgan_eps0.999.csv
eps=1.0    raw=sample_size_sweep/raw_scores_dpgan_1000_2500.csv
           effeps=sample_size_sweep/dpgan/1000_2500/effeps_dpgan_1000_2500.csv
eps=1.001  raw=extras/dpgan_spike_diagnosis/privacy/raw_scores_dpgan_eps1.001.csv
           effeps=extras/dpgan_spike_diagnosis/privacy/eps1.001/effeps_dpgan_eps1.001.csv
eps=10.0   raw=eps_sweep/dpgan/raw_scores_dpgan_eps10.csv
           effeps=eps_sweep/dpgan/eps10/effeps_dpgan_eps10.csv
eps=100.0  raw=eps_sweep/dpgan/raw_scores_dpgan_eps100.csv
           effeps=eps_sweep/dpgan/eps100/effeps_dpgan_eps100.csv


### Cell 2: load saved files
**Sources:** raw-score CSVs, per-attack eff_eps CSVs
- raw: `attack, ground_truth, raw_score`
- effeps: `attack, eps_low_90/95/99, eps_high_90/95/99`

In [2]:
def short(label: str) -> str:
    """'ClosestDistance(L_2, accuracy)' -> 'ClosestDistance' (names differ only in the suffix)."""
    return label.split("(")[0]

raw, saved = {}, {}
for eps, (rp, ep) in SOURCES.items():
    r = pd.read_csv(RES / rp)
    s = pd.read_csv(RES / ep)
    r["a"], s["a"] = r["attack"].map(short), s["attack"].map(short)
    raw[eps], saved[eps] = r, s
    print(f"eps={eps:<6} raw rows={len(r):<6} attacks={r['a'].nunique()}  effeps rows={len(s)}")

eps=0.1    raw rows=12500  attacks=5  effeps rows=5
eps=0.999  raw rows=12500  attacks=5  effeps rows=5
eps=1.0    raw rows=12500  attacks=5  effeps rows=5
eps=1.001  raw rows=12500  attacks=5  effeps rows=5
eps=10.0   raw rows=12500  attacks=5  effeps rows=5
eps=100.0  raw rows=12500  attacks=5  effeps rows=5


### Cell 3: recompute with TAPAS `EffectiveEpsilonReport`
**Sources:** 
- `raw` (raw-score CSVs from Cell 1). 
- Per attack, rows are taken in file order, as `summary.scores/labels` were.

Recomputation:
- `validation_split = min(0.5, max(0.1, 15 / n))` (same as `common.run_attack`); validation = `[:int(split*n)]`, test = the rest.
- `(threshold = tau, orientation = inverse true/false)` come from `EffectiveEpsilonReport._select_attack()` on the validation slice.
- Normal rule: positive class = label True, counted when `score >= tau`. Inverse rule: positive class = label False, counted when `score <= tau`.
- eps_low/eps_high come from the report's own `_estimate_effective_epsilon` on the test slice.

In [ ]:
def audit_one(scores, labels):
    n = len(scores)
    split = min(0.5, max(0.1, 15 / n))
    report = EffectiveEpsilonReport([SimpleNamespace(scores=scores, labels=labels)],
                                    validation_split=split, confidence_levels=CONF)
    ''' Tie-break (TAPAS 1.0.0 report.py::_select_attack_cp): 
    (threshold, orientation) with highest validation eps_low_90 
    - Tries tau ascending, normal before inverse orientation
    - Must be strictly greater validation eps_low_90 to replace the best 
    '''
    sel = report._select_attack()
    k = int(split * n)
    out = dict(n=n, n_val=k, n_test=n - k, split=split)
    if sel is None:
        return {**out, "tau": np.nan, "inverse": None, "rule": "none selected",
                "P": 0, "N": 0, "TP": 0, "FP": 0, **{f"{b}_{c}": np.nan for b in ("eps_low", "eps_high") for c in (90, 95, 99)}}
    _, tau, inverse = sel
    ts, tl = scores[k:], labels[k:]
    pos_label = not inverse
    hit = (lambda x: x <= tau) if inverse else (lambda x: x >= tau)
    P, N = int((tl == pos_label).sum()), int((tl != pos_label).sum())
    TP = int(hit(ts[tl == pos_label]).sum())
    FP = int(hit(ts[tl != pos_label]).sum())
    rule = ("inverse: score <= tau, label False = positive" if inverse
            else "normal: score >= tau, label True = positive")
    out.update(tau=float(tau), inverse=bool(inverse), rule=rule, P=P, N=N, TP=TP, FP=FP)
    for c in CONF:
        lo, hi = report._estimate_effective_epsilon(ts, tl, tau, c, inverse)
        out[f"eps_low_{round(c*100)}"], out[f"eps_high_{round(c*100)}"] = lo, hi
    return out

rows = []
for eps, r in raw.items():
    order = list(saved[eps]["a"])               # saved-file attack order (used for tie-breaking, like idxmax)
    for att in order:
        g = r[r["a"] == att]
        res = audit_one(g["raw_score"].to_numpy(float), g["ground_truth"].to_numpy().astype(bool))
        rows.append(dict(epsilon=eps, attack=att, **res))
R = pd.DataFrame(rows)
R["TP/P"] = R.TP.astype(str) + "/" + R.P.astype(str)
R["FP/N"] = R.FP.astype(str) + "/" + R.N.astype(str)
print(f"{len(R)} rows recomputed ({R.epsilon.nunique()} epsilons x {R.attack.nunique()} attacks)")

30 rows recomputed (6 epsilons x 5 attacks)


### Cell 4: summary table, worst-case attack per epsilon
**Sources:** recomputed table `R` (from the raw-score CSVs in Cell 1); attack order for ties follows the saved effeps CSV order.

- Worst case = highest recomputed `eps_low_95` (`idxmax`, as the repo's aggregation does). 
- `n_tied` is how many attacks share that maximum (all five tie at 0 when no attack certifies any epsilon).

In [ ]:
COLS = ["epsilon", "attack", "tau", "rule", "TP/P", "FP/N",
        "eps_low_90", "eps_high_90", "eps_low_95", "eps_high_95", "eps_low_99", "eps_high_99"]
worst = []
for eps, g in R.groupby("epsilon", sort=False):
    # Tie-break (manual, mirrors the repo's idxmax): ties on eps_low_95 go to the first attack in the saved effeps file order. 
    w = g.loc[g["eps_low_95"].idxmax()].copy()
    w["n_tied"] = int((g["eps_low_95"] == g["eps_low_95"].max()).sum())
    worst.append(w)
W = pd.DataFrame(worst)
W[COLS + ["n_tied"]].reset_index(drop=True)

,epsilon,attack,tau,rule,TP/P,FP/N,eps_low_90,eps_high_90,eps_low_95,eps_high_95,eps_low_99,eps_high_99,n_tied
0,0.1000,Groundhog,0.5700,"normal: score >= tau, label True = positive",202/1125,179/1125,0.0000,0.3857,0.0000,0.4231,0.0000,0.4985,5
1,0.9990,Groundhog,0.3400,"inverse: score <= tau, label False = positive",240/1125,11/1125,2.3875,3.8868,2.3032,4.0091,2.1384,4.2651,1
2,1.0000,Groundhog,0.6300,"normal: score >= tau, label True = positive",313/1125,33/1125,1.8179,2.7122,1.7613,2.7802,1.6493,2.9195,1
3,1.0010,Groundhog,0.6200,"normal: score >= tau, label True = positive",378/1125,32/1125,2.0446,2.9261,1.9893,2.9935,1.8799,3.1317,1
4,10.0000,Groundhog,0.2400,"normal: score >= tau, label True = positive",1125/1125,1122/1125,0.0000,0.0078,0.0000,0.0087,0.0000,0.0106,5
5,100.0000,Groundhog,0.3000,"normal: score >= tau, label True = positive",1122/1125,1121/1125,0.0000,0.0086,0.0000,0.0096,0.0000,0.0118,5


### Cell 5: one table per epsilon (all five attacks)
**Sources:** recomputed table `R` (raw-score CSVs from Cell 1). Same columns as the summary table; `P` and `N` are the test-slice class sizes for the chosen orientation.

In [5]:
from IPython.display import display, Markdown
for eps, g in R.groupby("epsilon", sort=False):
    src = SOURCES[eps][0]
    display(Markdown(f"#### epsilon = {eps}   (source: `results/{src}`; test slice n = {g.n_test.iloc[0]}, validation n = {g.n_val.iloc[0]}, split = {g.split.iloc[0]:.3f})"))
    display(g[COLS].reset_index(drop=True))

#### epsilon = 0.1   (source: `results/eps_sweep/dpgan/raw_scores_dpgan_eps0.1.csv`; test slice n = 2250, validation n = 250, split = 0.100)

,epsilon,attack,tau,rule,TP/P,FP/N,eps_low_90,eps_high_90,eps_low_95,eps_high_95,eps_low_99,eps_high_99
0,0.1000,Groundhog,0.5700,"normal: score >= tau, label True = positive",202/1125,179/1125,0.0000,0.3857,0.0000,0.4231,0.0000,0.4985
1,0.1000,ShadowModelling,0.1800,"normal: score >= tau, label True = positive",1123/1125,1125/1125,0.0000,0.0031,0.0000,0.0037,0.0000,0.0053
2,0.1000,LocalNeighbourhood,0.0000,"normal: score >= tau, label True = positive",1125/1125,1125/1125,0.0000,0.0033,0.0000,0.0039,0.0000,0.0053
3,0.1000,ProbabilityEstimation,-86.4915,"normal: score >= tau, label True = positive",1121/1125,1121/1125,0.0000,0.0081,0.0000,0.0093,0.0000,0.0116
4,0.1000,ClosestDistance,-2.1395,"normal: score >= tau, label True = positive",1124/1125,1125/1125,0.0000,0.0033,0.0000,0.0039,0.0000,0.0053


#### epsilon = 0.999   (source: `results/extras/dpgan_spike_diagnosis/privacy/raw_scores_dpgan_eps0.999.csv`; test slice n = 2250, validation n = 250, split = 0.100)

,epsilon,attack,tau,rule,TP/P,FP/N,eps_low_90,eps_high_90,eps_low_95,eps_high_95,eps_low_99,eps_high_99
0,0.9990,Groundhog,0.3400,"inverse: score <= tau, label False = positive",240/1125,11/1125,2.3875,3.8868,2.3032,4.0091,2.1384,4.2651
1,0.9990,ShadowModelling,0.2700,"inverse: score <= tau, label False = positive",57/1125,15/1125,0.5669,2.1643,0.4673,2.2853,0.2692,2.5347
2,0.9990,LocalNeighbourhood,0.0000,"normal: score >= tau, label True = positive",1125/1125,1125/1125,0.0000,0.0033,0.0000,0.0039,0.0000,0.0053
3,0.9990,ProbabilityEstimation,-89.2327,"normal: score >= tau, label True = positive",1124/1125,1123/1125,0.0000,0.0064,0.0000,0.0072,0.0000,0.0090
4,0.9990,ClosestDistance,-3.2344,"normal: score >= tau, label True = positive",1125/1125,1125/1125,0.0000,0.0033,0.0000,0.0039,0.0000,0.0053


#### epsilon = 1.0   (source: `results/sample_size_sweep/raw_scores_dpgan_1000_2500.csv`; test slice n = 2250, validation n = 250, split = 0.100)

...

,epsilon,attack,tau,rule,TP/P,FP/N,eps_low_90,eps_high_90,eps_low_95,eps_high_95,eps_low_99,eps_high_99
0,1.0010,Groundhog,0.6200,"normal: score >= tau, label True = positive",378/1125,32/1125,2.0446,2.9261,1.9893,2.9935,1.8799,3.1317
1,1.0010,ShadowModelling,0.2800,"inverse: score <= tau, label False = positive",60/1125,22/1125,0.3292,1.7121,0.2396,1.8141,0.0606,2.0227
2,1.0010,LocalNeighbourhood,0.0000,"normal: score >= tau, label True = positive",1125/1125,1125/1125,0.0000,0.0033,0.0000,0.0039,0.0000,0.0053
3,1.0010,ProbabilityEstimation,-89.2290,"normal: score >= tau, label True = positive",1124/1125,1123/1125,0.0000,0.0064,0.0000,0.0072,0.0000,0.0090
4,1.0010,ClosestDistance,-3.2273,"normal: score >= tau, label True = positive",1125/1125,1125/1125,0.0000,0.0033,0.0000,0.0039,0.0000,0.0053


#### epsilon = 10.0   (source: `results/eps_sweep/dpgan/raw_scores_dpgan_eps10.csv`; test slice n = 2250, validation n = 250, split = 0.100)

,epsilon,attack,tau,rule,TP/P,FP/N,eps_low_90,eps_high_90,eps_low_95,eps_high_95,eps_low_99,eps_high_99
0,10.0000,Groundhog,0.2400,"normal: score >= tau, label True = positive",1125/1125,1122/1125,0.0000,0.0078,0.0000,0.0087,0.0000,0.0106
1,10.0000,ShadowModelling,0.4900,"inverse: score <= tau, label False = positive",556/1125,537/1125,0.0000,0.1569,0.0000,0.1742,0.0000,0.2090
2,10.0000,LocalNeighbourhood,0.0000,"normal: score >= tau, label True = positive",1125/1125,1125/1125,0.0000,0.0033,0.0000,0.0039,0.0000,0.0053
3,10.0000,ProbabilityEstimation,-90.1666,"normal: score >= tau, label True = positive",1123/1125,1123/1125,0.0000,0.0062,0.0000,0.0071,0.0000,0.0089
4,10.0000,ClosestDistance,-3.5434,"normal: score >= tau, label True = positive",1119/1125,1111/1125,0.0000,0.0190,0.0000,0.0207,0.0000,0.0240


#### epsilon = 100.0   (source: `results/eps_sweep/dpgan/raw_scores_dpgan_eps100.csv`; test slice n = 2250, validation n = 250, split = 0.100)

,epsilon,attack,tau,rule,TP/P,FP/N,eps_low_90,eps_high_90,eps_low_95,eps_high_95,eps_low_99,eps_high_99
0,100.0000,Groundhog,0.3000,"normal: score >= tau, label True = positive",1122/1125,1121/1125,0.0000,0.0086,0.0000,0.0096,0.0000,0.0118
1,100.0000,ShadowModelling,0.2600,"normal: score >= tau, label True = positive",1121/1125,1115/1125,0.0000,0.0155,0.0000,0.0168,0.0000,0.0197
2,100.0000,LocalNeighbourhood,0.0000,"normal: score >= tau, label True = positive",1125/1125,1125/1125,0.0000,0.0033,0.0000,0.0039,0.0000,0.0053
3,100.0000,ProbabilityEstimation,-90.2266,"normal: score >= tau, label True = positive",1118/1125,1119/1125,0.0000,0.0091,0.0000,0.0105,0.0000,0.0133
4,100.0000,ClosestDistance,-3.8196,"normal: score >= tau, label True = positive",1125/1125,1125/1125,0.0000,0.0033,0.0000,0.0039,0.0000,0.0053


### Cell 6: check recomputed vs saved
**Sources:** recomputed `R`, and the saved per-attack effeps CSVs listed in Cell 1
(`eps_sweep/dpgan/eps0.1/…`, `extras/dpgan_spike_diagnosis/privacy/eps0.999/…`, `sample_size_sweep/dpgan/1000_2500/…`,
`extras/dpgan_spike_diagnosis/privacy/eps1.001/…`, `eps_sweep/dpgan/eps10/…`, `eps_sweep/dpgan/eps100/…`).
Columns compared: `eps_low_90/95/99` and `eps_high_90/95/99`, tolerance 1e-9. Infinite values must match exactly. The `effective_epsilon_<attack>.csv` files are not used.

In [6]:
TOL = 1e-9
BOUND_COLS = [f"{b}_{c}" for b in ("eps_low", "eps_high") for c in (90, 95, 99)]

def same(a, b):
    if np.isinf(a) or np.isinf(b):
        return a == b
    return bool(abs(a - b) <= TOL)

checks = []
for _, row in R.iterrows():
    s = saved[row.epsilon]
    s = s[s["a"] == row.attack]
    assert len(s) == 1, (row.epsilon, row.attack)
    s = s.iloc[0]
    ok_cols = {c: same(float(row[c]), float(s[c])) for c in BOUND_COLS}
    checks.append(dict(epsilon=row.epsilon, attack=row.attack,
                       result="PASS" if all(ok_cols.values()) else "FAIL",
                       max_abs_diff=max(abs(float(row[c]) - float(s[c])) if not (np.isinf(row[c]) or np.isinf(s[c])) else 0.0
                                        for c in BOUND_COLS),
                       failing=[c for c, v in ok_cols.items() if not v]))
CH = pd.DataFrame(checks)
for _, c in CH.iterrows():
    print(f"{c.result}  eps={c.epsilon:<6} {c.attack:<22} max|diff|={c.max_abs_diff:.2e}  {c.failing if c.failing else ''}")
n_pass = int((CH.result == "PASS").sum())
print(f"\n{n_pass}/{len(CH)} rows PASS")
assert n_pass == len(CH), "recomputed eps_low/eps_high differ from the saved per-attack effeps CSVs"

PASS  eps=0.1    Groundhog              max|diff|=3.33e-16  
PASS  eps=0.1    ShadowModelling        max|diff|=8.85e-17  
PASS  eps=0.1    LocalNeighbourhood     max|diff|=5.64e-17  
PASS  eps=0.1    ProbabilityEstimation  max|diff|=9.37e-17  
PASS  eps=0.1    ClosestDistance        max|diff|=9.63e-17  
PASS  eps=0.999  Groundhog              max|diff|=8.88e-16  
PASS  eps=0.999  ShadowModelling        max|diff|=4.44e-16  
PASS  eps=0.999  LocalNeighbourhood     max|diff|=5.64e-17  
PASS  eps=0.999  ProbabilityEstimation  max|diff|=8.85e-17  
PASS  eps=0.999  ClosestDistance        max|diff|=5.64e-17  
PASS  eps=1.0    Groundhog              max|diff|=8.88e-16  
PASS  eps=1.0    ShadowModelling        max|diff|=1.78e-15  
PASS  eps=1.0    LocalNeighbourhood     max|diff|=5.64e-17  
PASS  eps=1.0    ProbabilityEstimation  max|diff|=9.71e-17  
PASS  eps=1.0    ClosestDistance        max|diff|=5.29e-17  
PASS  eps=1.001  Groundhog              max|diff|=8.88e-16  
PASS  eps=1.001  ShadowM